# Benchmark Model - HAR-RV

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LinearRegression

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# Load train/test splits and prepare HAR-RV feature frame for model training and saving
DATA_DIR = Path("../data")
MODEL_DIR_HAR_RV = Path("artifacts/har_rv")
MODEL_DIR_HAR_RV.mkdir(parents=True, exist_ok=True)

def load_project_splits(data_dir=DATA_DIR):
    train_df = pd.read_csv(data_dir / "train_dataset.csv")
    train_df["date"] = pd.to_datetime(train_df["date"], format="%d/%m/%y", errors="raise")
    train_df = train_df.sort_values("date").reset_index(drop=True)

    test_df = pd.read_csv(data_dir / "test_dataset.csv")
    test_df["date"] = pd.to_datetime(test_df["date"], format="%Y-%m-%d", errors="raise")
    test_df = test_df.sort_values("date").reset_index(drop=True)

    return train_df, test_df

def build_model_base_frame(train_df, test_df):
    base_cols = ["date", "log_RV", "y_h1", "y_h3", "y_h5", "y_h7"]
    full_df = pd.concat([train_df[base_cols], test_df[base_cols]], ignore_index=True)
    return full_df.sort_values("date").reset_index(drop=True)

def build_har_rv_frame(base_df):
    har_df = base_df.copy()
    har_df["har_d"] = har_df["log_RV"].shift(1)
    har_df["har_w"] = har_df["log_RV"].rolling(window=5).mean().shift(1)
    har_df["har_m"] = har_df["log_RV"].rolling(window=22).mean().shift(1)

    feature_cols = ["har_d", "har_w", "har_m"]
    target_cols = ["y_h1", "y_h3", "y_h5", "y_h7"]
    har_df = har_df.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
    return har_df, feature_cols, target_cols

train_df, test_df = load_project_splits()
model_base_df = build_model_base_frame(train_df, test_df)
har_df, feature_cols, target_cols = build_har_rv_frame(model_base_df)

initial_start, initial_end = train_df["date"].min(), train_df["date"].max()
train_har = har_df[(har_df["date"] >= initial_start) & (har_df["date"] <= initial_end)].copy()
rolling_window = len(train_har)

print(
    f"Initial estimation sample: {train_har['date'].min().date()} -> "
    f"{train_har['date'].max().date()} ({len(train_har)} rows)"
)
print(f"Rolling window length used for saved models: {rolling_window} observations")

train_har[["date"] + feature_cols + target_cols].head()

Initial estimation sample: 2021-01-30 -> 2024-06-28 (1246 rows)
Rolling window length used for saved models: 1246 observations


,date,har_d,har_w,har_m,y_h1,y_h3,y_h5,y_h7
0,2021-01-30,-4.2090,-5.4214,-5.3629,-6.3489,-6.3514,-6.0472,-6.3634
1,2021-01-31,-5.7377,-5.3786,-5.4078,-6.1864,-6.4397,-6.6547,-6.0477
2,2021-02-01,-6.3489,-5.5085,-5.4404,-6.3514,-6.0472,-6.3634,-4.9954
3,2021-02-02,-6.1864,-5.6685,-5.4940,-6.4397,-6.6547,-6.0477,-5.3543
4,2021-02-03,-6.3514,-5.7667,-5.6204,-6.0472,-6.3634,-4.9954,-5.8478


In [3]:
def save_model_bundle(model, feature_cols, model_path, window_size, model_name):
    payload = {
        "model": model,
        "feature_cols": feature_cols,
        "window_size": window_size,
        "model_name": model_name,
    }
    with open(model_path, "wb") as handle:
        pickle.dump(payload, handle)


def load_model_bundle(model_path):
    with open(model_path, "rb") as handle:
        return pickle.load(handle)


def fit_model(train_frame, feature_cols, target_col, estimator):
    model = clone(estimator)
    model.fit(train_frame[feature_cols], train_frame[target_col])
    return model


def train_and_save_latest_rolling_models(
    full_frame, feature_cols, target_cols, window_size, estimator, model_dir, model_name
):
    model_dir.mkdir(parents=True, exist_ok=True)
    trained_models = {}
    latest_window = full_frame.iloc[-window_size:]

    for target in target_cols:
        target_train = latest_window.dropna(subset=feature_cols + [target])
        model = fit_model(target_train, feature_cols, target, estimator)
        save_model_bundle(model, feature_cols, model_dir / f"{target}.pkl", window_size, model_name)
        trained_models[target] = model

    return trained_models


def _coerce_prediction_input(raw_frame):
    if not isinstance(raw_frame, pd.DataFrame):
        raise TypeError("raw_frame must be a pandas DataFrame")

    required_cols = {"date", "log_RV"}
    missing_cols = sorted(required_cols - set(raw_frame.columns))
    if missing_cols:
        raise ValueError(f"raw_frame is missing required columns: {missing_cols}")

    feature_input = raw_frame.copy()
    feature_input["date"] = pd.to_datetime(feature_input["date"], errors="raise")
    feature_input = feature_input.sort_values("date").drop_duplicates(subset="date", keep="last")
    return feature_input.reset_index(drop=True)


def build_har_feature_frame_for_prediction(raw_frame):
    feature_input = _coerce_prediction_input(raw_frame)
    feature_input["har_d"] = feature_input["log_RV"].shift(1)
    feature_input["har_w"] = feature_input["log_RV"].rolling(window=5).mean().shift(1)
    feature_input["har_m"] = feature_input["log_RV"].rolling(window=22).mean().shift(1)

    feature_frame = feature_input.dropna(subset=["har_d", "har_w", "har_m"]).reset_index(drop=True)
    return feature_frame[["date", "log_RV", "har_d", "har_w", "har_m"]]


def predict_saved_har(feature_frame, model_dir, target_cols=None):
    if target_cols is None:
        target_cols = ["y_h1", "y_h3", "y_h5", "y_h7"]

    if feature_frame.empty:
        raise ValueError("feature_frame has no valid rows to score")

    prediction_table = feature_frame[["date"]].reset_index(drop=True).copy()

    for target in target_cols:
        bundle = load_model_bundle(model_dir / f"{target}.pkl")
        missing_features = sorted(set(bundle["feature_cols"]) - set(feature_frame.columns))
        if missing_features:
            raise ValueError(
                f"feature_frame is missing required model features for {target}: {missing_features}"
            )
        prediction_table[f"pred_{target}"] = bundle["model"].predict(feature_frame[bundle["feature_cols"]])

    return prediction_table


def predict_saved_har_from_raw(raw_frame, model_dir, feature_builder, target_cols=None, latest_only=True):
    feature_frame = feature_builder(raw_frame)
    if feature_frame.empty:
        raise ValueError(
            "Insufficient history to compute HAR features. Provide at least 23 dated log_RV observations."
        )

    prediction_table = predict_saved_har(feature_frame, model_dir, target_cols=target_cols)
    if latest_only:
        return prediction_table.tail(1).reset_index(drop=True)
    return prediction_table


def predict_har_rv(feature_frame, target_cols=None):
    return predict_saved_har(feature_frame, MODEL_DIR_HAR_RV, target_cols=target_cols)


def predict_har_rv_from_raw(raw_frame, target_cols=None, latest_only=True):
    return predict_saved_har_from_raw(
        raw_frame,
        MODEL_DIR_HAR_RV,
        build_har_feature_frame_for_prediction,
        target_cols=target_cols,
        latest_only=latest_only,
    )


model_specs = {
    "HAR-RV": {
        "estimator": LinearRegression(),
        "model_dir": MODEL_DIR_HAR_RV,
    },
}

saved_models = {}
coef_rows = []

for model_name, spec in model_specs.items():
    saved_models[model_name] = train_and_save_latest_rolling_models(
        har_df,
        feature_cols,
        target_cols,
        rolling_window,
        spec["estimator"],
        spec["model_dir"],
        model_name,
    )

    for target in target_cols:
        coef_rows.append(
            {
                "model": model_name,
                "target": target,
                "intercept": saved_models[model_name][target].intercept_,
                **dict(zip(feature_cols, saved_models[model_name][target].coef_)),
            }
        )

coef_df = pd.DataFrame(coef_rows).sort_values(["model", "target"]).reset_index(drop=True)

print(f"Saved HAR-RV models to: {MODEL_DIR_HAR_RV.resolve()}")
print("Latest model coefficients")
display(coef_df)

Saved HAR-RV models to: /Users/lijian/btc-rv-prediction/models/artifacts/har_rv
Latest model coefficients


,model,target,intercept,har_d,har_w,har_m
0,HAR-RV,y_h1,-2.4011,0.1219,0.2389,0.3322
1,HAR-RV,y_h3,-2.9707,-0.1323,0.7723,-0.0198
2,HAR-RV,y_h5,-3.1548,0.1651,0.3813,0.0506
3,HAR-RV,y_h7,-3.8353,0.3139,-0.2574,0.4538


## Extended Statistical Models
To keep the data source consistent with the benchmark section, the extended models below are built only from `train_dataset.csv` and `test_dataset.csv`. The target definition remains unchanged: each horizon predicts $y_{h}(t)=\log RV_{t+h}$.

Because the train/test datasets do not contain an intraday bipower-variation series, the jump term here is implemented as a **daily jump proxy** rather than an exact realised-jump decomposition. We define:
$$
J_t^{\text{proxy}} = \max\left(\log RV_t - \overline{\log RV}_{t}^{(5)}, 0\right),
$$
where $\overline{\log RV}_{t}^{(5)}$ is the same-day 5-day rolling average of log realised volatility.

- `HAR-RV-J`: adds the lagged daily jump proxy `jump_d`.
- `HAR-RV-J-H`: adds heterogeneous jump-proxy terms `jump_d`, `jump_w`, and `jump_m` so jump effects can vary across daily, weekly, and monthly scales.

Forecast evaluation and Diebold-Mariano testing are maintained separately in the `forecast evaluations` folder.

In [4]:
MODEL_DIR_HAR_RV_J = Path("artifacts/har_rv_j")
MODEL_DIR_HAR_RV_J_H = Path("artifacts/har_rv_j_h")
MODEL_DIR_HAR_RV_J.mkdir(parents=True, exist_ok=True)
MODEL_DIR_HAR_RV_J_H.mkdir(parents=True, exist_ok=True)


def build_jump_proxy_frame(base_df):
    jump_df = base_df[["date", "log_RV"]].copy()
    jump_df["jump_proxy"] = np.maximum(
        jump_df["log_RV"] - jump_df["log_RV"].rolling(window=5).mean(),
        0.0,
    )
    return jump_df[["date", "jump_proxy"]]


def build_extended_har_frame(base_df, jump_df):
    extended_df = base_df.merge(jump_df, on="date", how="left")
    extended_df["har_d"] = extended_df["log_RV"].shift(1)
    extended_df["har_w"] = extended_df["log_RV"].rolling(window=5).mean().shift(1)
    extended_df["har_m"] = extended_df["log_RV"].rolling(window=22).mean().shift(1)
    extended_df["jump_d"] = extended_df["jump_proxy"].shift(1)
    extended_df["jump_w"] = extended_df["jump_proxy"].rolling(window=5).mean().shift(1)
    extended_df["jump_m"] = extended_df["jump_proxy"].rolling(window=22).mean().shift(1)
    return extended_df


def build_extended_har_feature_frame_for_prediction(raw_frame):
    feature_input = _coerce_prediction_input(raw_frame)
    jump_df = build_jump_proxy_frame(feature_input)
    extended_df = build_extended_har_frame(feature_input[["date", "log_RV"]], jump_df)
    required_cols = ["har_d", "har_w", "har_m", "jump_d", "jump_w", "jump_m"]
    extended_df = extended_df.dropna(subset=required_cols).reset_index(drop=True)
    return extended_df[
        ["date", "log_RV", "jump_proxy", "har_d", "har_w", "har_m", "jump_d", "jump_w", "jump_m"]
    ]


def predict_har_rv_j(feature_frame, target_cols=None):
    return predict_saved_har(feature_frame, MODEL_DIR_HAR_RV_J, target_cols=target_cols)


def predict_har_rv_j_h(feature_frame, target_cols=None):
    return predict_saved_har(feature_frame, MODEL_DIR_HAR_RV_J_H, target_cols=target_cols)


def predict_har_rv_j_from_raw(raw_frame, target_cols=None, latest_only=True):
    return predict_saved_har_from_raw(
        raw_frame,
        MODEL_DIR_HAR_RV_J,
        build_extended_har_feature_frame_for_prediction,
        target_cols=target_cols,
        latest_only=latest_only,
    )


def predict_har_rv_j_h_from_raw(raw_frame, target_cols=None, latest_only=True):
    return predict_saved_har_from_raw(
        raw_frame,
        MODEL_DIR_HAR_RV_J_H,
        build_extended_har_feature_frame_for_prediction,
        target_cols=target_cols,
        latest_only=latest_only,
    )


jump_df = build_jump_proxy_frame(model_base_df)
extended_har_df = build_extended_har_frame(model_base_df, jump_df)

extended_model_specs = {
    "HAR-RV-J": {
        "estimator": LinearRegression(),
        "model_dir": MODEL_DIR_HAR_RV_J,
        "feature_cols": feature_cols + ["jump_d"],
    },
    "HAR-RV-J-H": {
        "estimator": LinearRegression(),
        "model_dir": MODEL_DIR_HAR_RV_J_H,
        "feature_cols": feature_cols + ["jump_d", "jump_w", "jump_m"],
    },
}

required_cols = target_cols + sorted({col for spec in extended_model_specs.values() for col in spec["feature_cols"]})
extended_har_df = extended_har_df.dropna(subset=required_cols).reset_index(drop=True)
if extended_har_df.empty:
    raise ValueError("No rows available for extended HAR model training after feature construction.")

extended_train_window = len(extended_har_df[extended_har_df["date"] <= initial_end])
extended_train_window = min(max(1, extended_train_window), len(extended_har_df))

extended_saved_models = {}
extended_coef_rows = []

for model_name, spec in extended_model_specs.items():
    extended_saved_models[model_name] = train_and_save_latest_rolling_models(
        extended_har_df,
        spec["feature_cols"],
        target_cols,
        extended_train_window,
        spec["estimator"],
        spec["model_dir"],
        model_name,
    )

    for target in target_cols:
        extended_coef_rows.append(
            {
                "model": model_name,
                "target": target,
                "intercept": extended_saved_models[model_name][target].intercept_,
                **dict(zip(spec["feature_cols"], extended_saved_models[model_name][target].coef_)),
            }
        )

extended_coef_df = pd.DataFrame(extended_coef_rows).sort_values(["model", "target"]).reset_index(drop=True)

print(f"Saved HAR-RV-J models to:   {MODEL_DIR_HAR_RV_J.resolve()}")
print(f"Saved HAR-RV-J-H models to: {MODEL_DIR_HAR_RV_J_H.resolve()}")
print("Latest extended-model coefficients")
display(extended_coef_df)

Saved HAR-RV-J models to:   /Users/lijian/btc-rv-prediction/models/artifacts/har_rv_j
Saved HAR-RV-J-H models to: /Users/lijian/btc-rv-prediction/models/artifacts/har_rv_j_h
Latest extended-model coefficients


,model,target,intercept,har_d,har_w,har_m,jump_d,jump_w,jump_m
0,HAR-RV-J,y_h1,-2.2114,-0.2917,0.6451,0.3947,0.8580,NaN,NaN
1,HAR-RV-J,y_h3,-2.9328,-0.2428,0.8808,-0.0047,0.2286,NaN,NaN
2,HAR-RV-J,y_h5,-3.1144,0.0686,0.4762,0.0645,0.2000,NaN,NaN
3,HAR-RV-J,y_h7,-3.8263,0.4048,-0.3461,0.4457,-0.1882,NaN,NaN
4,HAR-RV-J-H,y_h1,-2.4009,-0.3416,0.9101,0.1403,1.0947,-1.0508,0.3997
5,HAR-RV-J-H,y_h3,-2.7341,-0.2558,1.0079,-0.1461,0.3096,0.0236,-1.5919
6,HAR-RV-J-H,y_h5,-2.8403,0.0823,0.4555,0.0670,0.1524,0.5544,-1.6282
7,HAR-RV-J-H,y_h7,-3.8809,0.3778,-0.1911,0.2932,-0.0561,-0.5122,-0.1125
